In [ ]:
"""
Evaluation Script with Rendering Window
========================================

Visualize a trained SAC agent performing the Lift task in Robosuite.

PREREQUISITES (Docker on Windows):
----------------------------------
1. Install VcXsrv on Windows: https://sourceforge.net/projects/vcxsrv/

2. Launch XLaunch with these settings:
   - Multiple windows
   - Start no client
   - CHECK "Disable access control" (important!)

3. Set DISPLAY in Docker (add to docker-compose.yml or export manually):
   export DISPLAY=host.docker.internal:0.0

4. Ensure the container was started AFTER setting DISPLAY

USAGE:
------
    cd /LearnFlake/src/rl_autonomy/rl_agent
    python evaluate_render.py

TROUBLESHOOTING:
----------------
- "Failed to open display": VcXsrv not running or DISPLAY not set
- "Could not initialize GLFW": Try `export MUJOCO_GL=glx` before running
- Black window: Check VcXsrv firewall permissions
"""

import os
import glob

# Auto-detect X11 display for WSL2 / Docker (avoids silent GLFW segfault)
_x11_sockets = glob.glob('/tmp/.X11-unix/X*')
if _x11_sockets:
    _display_num = _x11_sockets[0].replace('/tmp/.X11-unix/X', '')
    os.environ.setdefault('DISPLAY', f':{_display_num}')
else:
    # Docker-on-Windows or no local X11 — try host X server (VcXsrv / X410)
    os.environ.setdefault('DISPLAY', 'host.docker.internal:0.0')

os.environ['MUJOCO_GL'] = 'glfw'

print(f"DISPLAY:   {os.environ.get('DISPLAY', 'NOT SET')}")
print(f"MUJOCO_GL: {os.environ.get('MUJOCO_GL')}")

In [2]:
import torch
if torch.cuda.is_available():
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

GPU Memory: 6.44 GB
GPU Memory Allocated: 0.00 GB


In [ ]:
import time
import numpy as np
from gym_wrapper import RobosuiteGymWrapper
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor


In [4]:

# =============================================================================
# CONFIGURATION - Modify these as needed
# =============================================================================
MODEL_PATH = "sac_Rover2026_V1_model.zip"
N_EPISODES = 5
ROBOT = "Rover2026"


In [ ]:

# =============================================================================
# ENVIRONMENT SETUP
# =============================================================================
print("Creating environment with renderer enabled...")
env = RobosuiteGymWrapper(
    "Lift",
    robots=ROBOT,
    has_renderer=True,
    has_offscreen_renderer=False,
    use_camera_obs=False,
    reward_shaping=True
)
env = Monitor(env)


In [ ]:

# =============================================================================
# LOAD MODEL
# =============================================================================
print(f"Loading model from: {MODEL_PATH}")

if not os.path.exists(MODEL_PATH):
    print(f"Model not found at {MODEL_PATH}. Train first!")
    env.close()
    exit(0)

model = SAC.load(MODEL_PATH, env=env, device="cpu")
print(f"Model trained for {model.num_timesteps} timesteps")


Loading model from: sac_Rover2026_V1_model.zip
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Model trained for 1001000 timesteps


: 

In [ ]:

# =============================================================================
# RUN EVALUATION WITH RENDERING
# =============================================================================
# Uses a manual loop instead of evaluate_policy(render=True) so that:
# - Robosuite's internal render (triggered on each step when has_renderer=True)
#   is the only render path — no double-render / double-context crash.
# - Episode progress is visible in real time.
print(f"\nRunning {N_EPISODES} evaluation episodes with rendering...")
print("=" * 50)

start_time = time.time()
rewards = []

for ep in range(N_EPISODES):
    obs, _ = env.reset()
    ep_reward = 0.0
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        ep_reward += reward
        done = terminated or truncated
    rewards.append(ep_reward)
    print(f"Episode {ep + 1}/{N_EPISODES}: reward = {ep_reward:.2f}")

elapsed = time.time() - start_time

print("=" * 50)
print(f"Mean reward: {np.mean(rewards):.2f}, Std reward: {np.std(rewards):.2f}")
print(f"Evaluation took {elapsed:.2f} seconds")

env.close()
print("Evaluation complete!")
